In [ ]:
# Analisar os dados de descontos do nosso banco de dados de vendas da Amazon para ajudar a equipe de Marketing e Vendas a entender nossa política de preços. 
# Seu objetivo é determinar se os descontos médios que oferecemos são consistentes ou se existem padrões incomuns que precisam de atenção.
# Analisar Dados para explorar a coluna de desconto no conjunto de dados e criar um relatório 
# De modo a destacar suas descobertas sobre nossas práticas de desconto.

%pip install pandas
import pandas as pd
import numpy as np

vendas = pd.read_csv('01.amazon_sales_dataset.csv')
vendas.columns

Note: you may need to restart the kernel to use updated packages.


Index(['order_id', 'order_date', 'ship_date', 'delivery_date', 'order_status',
       'customer_id', 'customer_name', 'country', 'state', 'city',
       'product_id', 'product_name', 'category', 'sub_category', 'brand',
       'quantity', 'unit_price', 'discount', 'shipping_cost', 'total_sales',
       'payment_method'],
      dtype='str')

In [17]:
vendas.dtypes

order_id              str
order_date            str
ship_date             str
delivery_date         str
order_status          str
customer_id           str
customer_name         str
country               str
state                 str
city                  str
product_id            str
product_name          str
category              str
sub_category          str
brand                 str
quantity            int64
unit_price        float64
discount          float64
shipping_cost     float64
total_sales       float64
payment_method        str
dtype: object

In [18]:
vendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        10000 non-null  str    
 1   order_date      10000 non-null  str    
 2   ship_date       10000 non-null  str    
 3   delivery_date   10000 non-null  str    
 4   order_status    10000 non-null  str    
 5   customer_id     10000 non-null  str    
 6   customer_name   10000 non-null  str    
 7   country         10000 non-null  str    
 8   state           10000 non-null  str    
 9   city            10000 non-null  str    
 10  product_id      10000 non-null  str    
 11  product_name    10000 non-null  str    
 12  category        10000 non-null  str    
 13  sub_category    10000 non-null  str    
 14  brand           10000 non-null  str    
 15  quantity        10000 non-null  int64  
 16  unit_price      10000 non-null  float64
 17  discount        10000 non-null  float64
 18

In [19]:
coluna_alvo = 'discount'
# Medidas de Tendências Centrais
total_vendas_desconto = vendas[coluna_alvo].sum()
media_valor_vendas_desconto = vendas[coluna_alvo].mean()
venda_minima_desconto = vendas[coluna_alvo].min()
venda_maxima_desconto = vendas[coluna_alvo].max()
print(f'venda_maxima_desconto: {venda_maxima_desconto}')
print(f'venda_minima_desconto: {venda_minima_desconto}')
print(f'media_valor_vendas_desconto: {media_valor_vendas_desconto}')

venda_maxima_desconto: 0.3
venda_minima_desconto: 0.0
media_valor_vendas_desconto: 0.14996800000000002


In [20]:
# Medidas de Posição
q1 = np.percentile(vendas[coluna_alvo], 25)
q2 = np.percentile(vendas[coluna_alvo], 50)
# q2 Mediana
q3 = np.percentile(vendas[coluna_alvo], 75)

In [21]:
# Avaliar a confiabilidade da média
delta_media_mediana = media_valor_vendas_desconto - q2
distancia_percentual = media_valor_vendas_desconto / q2
# Caso 1: Distancia <= 10% -> (0.9 ~ 1.1) Média é altamente confiável
# Caso 2: Distancia entre 10% e 25% -> (0.75 ~0.9 / 1.1 ~ 1.25) Média é moderadamente confiável
# Caso 3: Distancia > 25% -> (0.75 ~ 1.25) Média é pouco confiável, ela deve ser distorcida nos valores extremos, ou seja, outliers.

In [22]:
# Assimetria (Skewness)
assimetria = vendas[coluna_alvo].skew()
# Caso 1: Skewness entre -0.5 e 0.5 -> Distribuição aproximadamente simétrica R = equilibrada
# Caso 2: Skewness > 0.5 (Positiva) R = Valores altos são raros, porém são tão grandes que distorcem o cálculo
# Caso 3: Skewness < -0.5 (Negativa) R = Valores baixos são raros, porém são tão grandes que distorcem o cálculo

In [30]:
print ('\n --- Resumo Executivo de Descontos ---')

print(f'2. Gasto Médio:{media_valor_vendas_desconto:,.2f}')
# (story telling): Valor médio que um cliente obteve de desconto conosco.
# Média de Valor das Vendas com Descontos: {media_valor_vendas_desconto}')
print(f'3. Maior Venda:{venda_maxima_desconto:,.2f}')
# (story telling): Nosso Desconto Record. investigar o perfil do cliente, da categoria do produto ou da época do ano para entender o contexto dessa transação. 
print(f'4. Menor Venda:{venda_minima_desconto:,.2f}')
# (story telling): Investigar se há produtos pcom potencial para incrementar suas vendas se um desconto for aplicado.


 --- Resumo Executivo de Descontos ---
2. Gasto Médio:0.15
3. Maior Venda:0.30
4. Menor Venda:0.00


In [ ]:
print('--- Relatório Estatístico de Valores de Descontos ---')
print(f'A Média é $ {media_valor_vendas_desconto:,.2f}') 
print(f'A mediana (O Valor Central) é {q2:,.2f}')
print(f' O Delta (A Diferença Financeira entre elas) é {delta_media_mediana:,.2f}')
print('\n --- Análise Comportamental ---')
print(f'Distancia Relativa entre Média e Mediana: {distancia_percentual:.2f}')
if abs(distancia_percentual-1) <= 0.1:
    print("INFERÊNCIA: Baixa Dispersão. A média é altamente confiável para representar a coluna.")
elif abs(distancia_percentual-1) <= 0.25:
    print("INFERÊNCIA: Dispersão Moderada. Atenção a valores extremos.")
else:
    print("INFERÊNCIA: Alta Dispersão. A média é não confiável para representar a coluna, considere utilizar a mediana.")

print(f"Grau de Assimetria (Skewness): {assimetria:.4f}")
if assimetria > 0.5:
    print("INFERÊNCIA: (Assimétrica positiva): cauda longa à direita.")
    # Story telling: Os grandes clientes são minoria, todavia faturam valores expressivos, ou seja, são os responsáveis por grande parte do faturamento.
elif assimetria < -0.5:
    print("INFERÊNCIA: (Assimétrica negativa): cauda longa à esquerda.")
    # Story telling: O valor médio está sendo puxado para baixo por um grande número de vendas com desconto de baixo valor, talvez liquidação.
else:
    print("INFERÊNCIA: (Simétrica): distribuição equilibrada. Média e mediana estão próximas, indicando uma distribuição equilibrada dos valores de vendas.")
 

--- Relatório Estatístico de Negócio ---
A Média é $ 0.15
A mediana (O Valor Central) é $ 0.15
 O Delta (A Diferença Financeira entre elas) é $ -0.00

 --- Análise Comportamental ---
Distancia Relativa entre Média e Mediana: 1.00
INFERÊNCIA: Baixa Dispersão. A média é altamente confiável para representar a coluna.
Grau de Assimetria (Skewness): -0.0042
INFERÊNCIA: (Simétrica): distribuição equilibrada. Média e mediana estão próximas, indicando uma distribuição equilibrada dos valores de vendas.


A distribuição dos descontos e sua variação parece estar coerente com as metas de vendas. 
Cabe investigar se a sazonabilidade influenciou no bom resultado das vendas, assim como agrupamentos de vendas por categorias.
Por fim, é interssante pesquisar se os produtos com baixas vendas e sem desconto poderiam ser impulsionados por aplicação de descontos